In [ ]:
# Setup: clone the StarX repo and the pinned TripoSR commit, install this
# notebook's dependencies, mount Drive, and report what machine we are on.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "05"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR, TRIPOSR_DIR = "/content/StarX", "/content/TripoSR"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if not os.path.exists(TRIPOSR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git",
         TRIPOSR_DIR],
        check=True,
    )
subprocess.run(["git", "-C", TRIPOSR_DIR, "checkout", "-q", TRIPOSR_COMMIT], check=True)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
if pins.PIP_PINS[NOTEBOOK_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

# 05 - Training

Everything meets here: the surgered model from notebook 04 learns to turn sketch stacks into 3D shapes, supervised purely by rendering - compare small rendered crops of its prediction against the ground-truth views from notebook 03.

One optimizer step: draw a few designs, encode each sketch stack into a triplane scene code (in fast bfloat16), render a couple of crops per design at ground-truth cameras (in full precision), score them with three loss terms, backpropagate in two stages, clip, update. Only the LoRA adapters and the new patch embedding move; 99% of the model stays frozen.

Requirements: the shards from notebook 03 (with the same SMOKE setting) and a GPU runtime - L4 or better. Set SMOKE to overfit a tiny set first: if the loss does not collapse there, something is wrong and the full run would waste hours.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import time
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from starx import cameras, checkpoint, data, shards, viz
from starx import model as smodel
from starx.config import FOVY_DEG, StarXConfig, run_dir, shard_dir

SMOKE = False    # True: overfit the 20-design smoke shards as a sanity run
RUN_NAME = "smoke_overfit" if SMOKE else "baseline_l4"
RESUME = True    # False: ignore existing checkpoints for this run name
SEED = 1337
SPLIT_PREFIX = "smoke_" if SMOKE else ""

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX")
    if DRIVE is not None
    else os.path.join(REPO_DIR, "data", "StarX"),
    local_root="/content/starx_local"
    if IN_COLAB
    else os.path.join(REPO_DIR, "data", "local"),
    # surgery - must match notebook 04
    max_sketch_channels=6,
    conv_init="i3d_mean",
    lora_r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    dino_layers=(0, 1, 2, 3),
    train_triplane_embed=False,
    # training
    total_steps=1000 if SMOKE else 20000,
    lr=1e-4,
    warmup_steps=100 if SMOKE else 500,
    accum_designs=4 if SMOKE else 8,
    views_per_design=2,
    render_crop=128,
    lambda_mse=1.0,
    lambda_lpips=2.0,
    lambda_mask=0.05,
    grad_clip=1.0,
    ckpt_every=250 if SMOKE else 500,
    keep_k=3,
    val_every=200 if SMOKE else 1000,
    gt_size=256,
    eval_chunk=131072,
    seed=SEED,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(
    f"run {RUN_NAME}: {cfg.total_steps} steps, "
    f"{cfg.accum_designs} designs x {cfg.views_per_design} views per step, "
    f"crop {cfg.render_crop}, device {device}"
)

In [ ]:
# Copy the shards from Drive to fast local disk (once per session), open
# the datasets, and browse one item end to end.
train_remote = shard_dir(cfg, f"{SPLIT_PREFIX}train")
val_remote = shard_dir(cfg, f"{SPLIT_PREFIX}test")
train_local = Path(cfg.local_root) / f"{SPLIT_PREFIX}train"
val_local = Path(cfg.local_root) / f"{SPLIT_PREFIX}test"
shards.prepare_local(train_remote, train_local, progress=tqdm)
shards.prepare_local(val_remote, val_local, progress=tqdm)

train_dataset = data.DesignDataset(train_local / "cache")
val_dataset = data.DesignDataset(val_local / "cache")
assert len(train_dataset) > 0, "no train shards found - run notebook 03 first (matching SMOKE)"
if len(val_dataset) == 0:
    val_dataset = train_dataset
    print("no test shards - using train designs for validation visuals")
print(f"train designs: {len(train_dataset)}   validation designs: {len(val_dataset)}")

browse = train_dataset[0]
fig = viz.show_sketch_stack(browse["stack_uint8"], browse["meta"], title=browse["design_id"])
plt.show()
fig = viz.show_view_grid(list(browse["views"][:3]), list(browse["masks"][:3]))
plt.show()

In [ ]:
# Build the surgered model exactly as notebook 04 verified it, then turn
# on gradient checkpointing (recompute activations during backward instead
# of storing them) and disable render chunking - under autograd the memory
# lever is crop size, not chunking.
model, build_info = smodel.build_starx_model(cfg, TRIPOSR_DIR, device=device)
model.image_tokenizer.model.gradient_checkpointing_enable()
model.backbone.gradient_checkpointing = True
model.renderer.set_chunk_size(0)

trainable_params = [p for p in model.parameters() if p.requires_grad]
totals = build_info["param_table"]["ALL"]
print(f"trainable: {totals['trainable']:,} of {totals['total']:,} "
      f"({100 * totals['trainable'] / totals['total']:.2f}%)")
print("lora modules:", build_info["lora_counts"])

In [ ]:
# Optimizer: AdamW with two groups (no weight decay on LoRA, a little on
# the new conv), linear warmup into a cosine decay, and the frozen VGG
# network that scores the perceptual loss.
from torchmetrics.image import LearnedPerceptualImagePatchSimilarity

conv_params = list(
    model.image_tokenizer.model.embeddings.patch_embeddings.projection.parameters()
)
conv_param_ids = {id(p) for p in conv_params}
lora_params = [p for p in trainable_params if id(p) not in conv_param_ids]
optimizer = torch.optim.AdamW(
    [
        {"params": lora_params, "weight_decay": 0.0},
        {"params": conv_params, "weight_decay": 0.01},
    ],
    lr=cfg.lr,
)


def lr_lambda(step):
    if step < cfg.warmup_steps:
        return (step + 1) / cfg.warmup_steps
    progress = (step - cfg.warmup_steps) / max(1, cfg.total_steps - cfg.warmup_steps)
    return 0.1 + 0.45 * (1 + np.cos(np.pi * min(progress, 1.0)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type="vgg", normalize=True)
lpips_metric = lpips_metric.to(device).requires_grad_(False)
lpips_metric.eval()
print(f"lora params: {sum(p.numel() for p in lora_params):,}   "
      f"conv params: {sum(p.numel() for p in conv_params):,}")

## The render path, and a two-stage backward

TripoSR ships a render method, but it is wrapped in no-gradient mode and composites over white without exposing opacity - unusable for training, where gradients must flow and the mask loss needs opacity. So the next cell writes the ray march ourselves from the renderer's building blocks. It is the one place the volume-rendering math is fully visible.

Memory is dominated by rendering, not by the transformer: one small crop already queries the NeRF at about two million points, and that graph is the VRAM peak. The training step therefore handles views one at a time - render a view, backpropagate its loss only as far as the scene code, free the render graph, next view - and only when a design's views are done does the collected scene-code gradient flow through the big encoder, once. Same gradients as one giant backward, at a fraction of the memory. The crop size is the single biggest memory knob if the step ever runs out.

In [ ]:
# The differentiable ray march (adapted from TriplaneNeRFRenderer._forward,
# which hides opacity and composites over white). For every ray: sample
# points between where it enters and leaves the scene box, query the
# triplane NeRF, and blend colors front to back with the classic
# alpha-compositing recursion.
from tsr.utils import rays_intersect_bbox


def render_rays(model, scene_code, rays_o, rays_d):
    """Returns (rgb_fg, opacity): foreground color and how solid each pixel
    is, before any background is composited."""
    renderer, decoder = model.renderer, model.decoder
    shape = rays_o.shape[:-1]
    rays_o, rays_d = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
    n_rays = rays_o.shape[0]

    t_near, t_far, valid = rays_intersect_bbox(rays_o, rays_d, renderer.cfg.radius)
    t_near, t_far = t_near[valid], t_far[valid]

    t_vals = torch.linspace(
        0, 1, renderer.cfg.num_samples_per_ray + 1, device=scene_code.device
    )
    t_mid = (t_vals[:-1] + t_vals[1:]) / 2.0
    z_vals = t_near * (1 - t_mid[None]) + t_far * t_mid[None]
    xyz = (
        rays_o[valid][:, None, :]
        + z_vals[..., None] * rays_d[valid][:, None, :]
    )

    out = renderer.query_triplane(decoder, xyz, scene_code)

    deltas = t_vals[1:] - t_vals[:-1]
    alpha = 1 - torch.exp(-deltas * out["density_act"][..., 0])
    transmittance = torch.cat(
        [
            torch.ones_like(alpha[:, :1]),
            torch.cumprod(1 - alpha[:, :-1] + 1e-10, dim=-1),
        ],
        dim=-1,
    )
    weights = alpha * transmittance
    rgb_valid = (weights[..., None] * out["color"]).sum(dim=-2)
    opacity_valid = weights.sum(dim=-1)

    rgb_fg = torch.zeros(n_rays, 3, dtype=rgb_valid.dtype, device=rgb_valid.device)
    opacity = torch.zeros(n_rays, dtype=opacity_valid.dtype, device=opacity_valid.device)
    rgb_fg[valid] = rgb_valid
    opacity[valid] = opacity_valid
    return rgb_fg.reshape(*shape, 3), opacity.reshape(*shape)


print("render_rays defined")

In [ ]:
# Where the loss looks: crop windows biased toward the object (most of a
# view is background, which teaches nothing).
demo_item = train_dataset[0]
crop_rng = np.random.default_rng(SEED)
box = data.sample_crop_box(demo_item["masks"][0], cfg.render_crop, crop_rng)
fig = viz.crop_debug_figure(demo_item["views"][0], demo_item["masks"][0], box)
plt.show()
print("crop box (top, left, h, w):", box)

In [ ]:
# Integration check: render that crop with the untrained model, next to
# the ground truth it will be compared against.
with torch.no_grad():
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
        demo_code = smodel.encode_sketches(model, demo_item["sketch"][None].to(device))[0]
    demo_code = demo_code.float()
    rays_o, rays_d = cameras.rays_for_crop(
        demo_item["c2ws"][0], FOVY_DEG, cfg.gt_size, box
    )
    rgb_fg, opacity = render_rays(model, demo_code, rays_o.to(device), rays_d.to(device))
print("rendered crop:", tuple(rgb_fg.shape), "  opacity:", tuple(opacity.shape))

top, left, h, w = box
gt_crop = demo_item["views"][0][top : top + h, left : left + w]
pred_crop = (rgb_fg + 0.5 * (1 - opacity[..., None])).clamp(0, 1).cpu().numpy()
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.4))
axes[0].imshow(gt_crop)
axes[0].set_title("ground-truth crop")
axes[1].imshow(pred_crop)
axes[1].set_title("untrained render")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
# The loss, term by term: pixel MSE and perceptual LPIPS on crops
# composited over the same mid-gray, plus a small mask term that suppresses
# floaters. Weights follow the TripoSR paper. Evaluated once, untrained.
def composite_over_gray(rgb_fg, opacity, bg=0.5):
    return rgb_fg + bg * (1.0 - opacity[..., None])


def compute_loss(rgb_fg, opacity, gt_rgb, gt_mask):
    gt = torch.where(gt_mask[..., None], gt_rgb, torch.full_like(gt_rgb, 0.5))
    pred = composite_over_gray(rgb_fg, opacity)
    mse = F.mse_loss(pred, gt)
    lpips_term = lpips_metric(
        pred.clamp(0, 1).permute(2, 0, 1)[None], gt.permute(2, 0, 1)[None]
    )
    mask_term = F.binary_cross_entropy(
        opacity.clamp(1e-4, 1 - 1e-4), gt_mask.float()
    )
    total = (
        cfg.lambda_mse * mse
        + cfg.lambda_lpips * lpips_term
        + cfg.lambda_mask * mask_term
    )
    return total, {
        "mse": float(mse),
        "lpips": float(lpips_term),
        "mask": float(mask_term),
    }


gt_rgb = torch.from_numpy(np.ascontiguousarray(gt_crop)).float().to(device) / 255.0
gt_mask = torch.from_numpy(
    np.ascontiguousarray(demo_item["masks"][0][top : top + h, left : left + w])
).to(device)
loss, parts = compute_loss(rgb_fg, opacity, gt_rgb, gt_mask)
print(f"untrained loss: {float(loss):.3f}   terms: {parts}")

In [ ]:
# One optimizer step: a handful of designs, a few views each, the
# two-stage backward, clip, update. Every random choice is seeded by the
# step number, which is what makes resuming exact.
def train_step(step):
    model.train()
    indices = data.draw_design_indices(
        step, cfg.accum_designs, len(train_dataset), cfg.seed
    )
    n_terms = cfg.accum_designs * cfg.views_per_design
    totals = {"loss": 0.0, "mse": 0.0, "lpips": 0.0, "mask": 0.0}
    for index in indices:
        item = train_dataset[index]
        sketch = item["sketch"][None].to(device)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
            scene_code = smodel.encode_sketches(model, sketch)[0]
        scene_code = scene_code.float()
        code_leaf = scene_code.detach().requires_grad_(True)

        view_ids = data.choose_views(
            step, item["design_id"], item["meta"]["n_views"],
            cfg.views_per_design, cfg.seed,
        )
        for v in view_ids:
            crop_rng = np.random.default_rng(
                zlib.crc32(f"{item['design_id']}:{step}:{v}".encode())
            )
            box = data.sample_crop_box(item["masks"][v], cfg.render_crop, crop_rng)
            top, left, h, w = box
            rays_o, rays_d = cameras.rays_for_crop(
                item["c2ws"][v], FOVY_DEG, cfg.gt_size, box
            )
            rgb_fg, opacity = render_rays(
                model, code_leaf, rays_o.to(device), rays_d.to(device)
            )
            gt_rgb = torch.from_numpy(
                np.ascontiguousarray(item["views"][v][top : top + h, left : left + w])
            ).float().to(device) / 255.0
            gt_mask = torch.from_numpy(
                np.ascontiguousarray(item["masks"][v][top : top + h, left : left + w])
            ).to(device)
            loss, parts = compute_loss(rgb_fg, opacity, gt_rgb, gt_mask)
            (loss / n_terms).backward()  # frees this view's render graph
            totals["loss"] += float(loss) / n_terms
            for key in ("mse", "lpips", "mask"):
                totals[key] += parts[key] / n_terms

        # stage two: push the accumulated scene-code gradient through the
        # encoder once for this design
        scene_code.backward(gradient=code_leaf.grad)

    grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, cfg.grad_clip)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    totals["grad_norm"] = float(grad_norm)
    totals["lr"] = scheduler.get_last_lr()[0]
    return totals


print("train_step defined - the loop below times its first call")

## Surviving disconnects

Colab sessions end - idle timeouts, usage caps, browser crashes. The run treats that as normal:

- every checkpoint interval, the trainable tensors (a few dozen MB, not the 1.7 GB model), optimizer state, schedule, and all random-number states are written to Drive atomically (temp file, then rename - a mid-write disconnect can never corrupt the newest checkpoint), keeping only the last few;
- the step number seeds every random choice (which designs, which views, which crops), so a resumed run draws exactly what an uninterrupted run would have drawn - no dataloader state to save;
- logs are append-only, so the loss curve just continues.

Killing the loop and re-running the notebook with the same run name is the officially supported way to train for longer than one session.

In [ ]:
# Resume: restore the newest checkpoint for this run name - model,
# optimizer, schedule, and random state - or start fresh.
rdir = run_dir(cfg, RUN_NAME)
latest = checkpoint.find_latest(rdir) if RESUME else None
if latest is not None:
    ckpt_path, start_step = latest
    state = checkpoint.load_checkpoint(ckpt_path)
    smodel.load_trainable_state_dict(model, state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    checkpoint.restore_rng(state["rng"])
    print(f"resumed {RUN_NAME} at step {start_step}")
else:
    start_step = 0
    print(f"starting {RUN_NAME} fresh")

In [ ]:
# The loop. Everything before was preparation; this cell can run for hours
# and can be killed and re-run at any time - it resumes automatically.
# After its first step it prints the step time and VRAM peak so a memory
# problem surfaces in seconds, not hours.
log_path = rdir / "logs" / "train_log.jsonl"
grid_dir = rdir / "val_grids"
grid_dir.mkdir(parents=True, exist_ok=True)
val_items = [val_dataset[i] for i in range(min(4, len(val_dataset)))]


def render_val_grid(items, step):
    """Sketch | ground truth | current prediction, one row per design."""
    model.eval()
    model.renderer.set_chunk_size(cfg.eval_chunk)
    fig, axes = plt.subplots(len(items), 3, figsize=(9.6, 3.1 * len(items)))
    axes = np.atleast_2d(axes)
    with torch.no_grad():
        for row, item in enumerate(items):
            code = smodel.encode_sketches(model, item["sketch"][None].to(device))[0]
            code = code.float()
            rays_o, rays_d = cameras.rays_full(item["c2ws"][0], FOVY_DEG, cfg.gt_size)
            rgb_fg, opacity = render_rays(model, code, rays_o.to(device), rays_d.to(device))
            pred = composite_over_gray(rgb_fg, opacity).clamp(0, 1).cpu().numpy()
            gt = item["views"][0].astype(np.float32) / 255.0
            gt = np.where(item["masks"][0][..., None], gt, 0.5)
            axes[row, 0].imshow(item["stack_uint8"][0], cmap="gray", vmin=0, vmax=255)
            axes[row, 1].imshow(gt)
            axes[row, 2].imshow(pred)
            axes[row, 0].set_ylabel(item["design_id"][:12], fontsize=8)
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0, 0].set_title("first sketch")
    axes[0, 1].set_title("ground truth")
    axes[0, 2].set_title(f"prediction @ step {step}")
    model.renderer.set_chunk_size(0)
    model.train()
    return fig


progress = tqdm(range(start_step, cfg.total_steps), initial=start_step, total=cfg.total_steps)
for step in progress:
    step_started = time.time()
    totals = train_step(step)
    if step == start_step:
        print(f"first step: {time.time() - step_started:.1f}s")
        if device == "cuda":
            print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
    progress.set_postfix(loss=f"{totals['loss']:.3f}")
    completed = step + 1
    if completed % 50 == 0 or completed == cfg.total_steps:
        checkpoint.append_log(log_path, {"step": completed, **totals})
    if completed % cfg.ckpt_every == 0 or completed == cfg.total_steps:
        checkpoint.save_checkpoint(
            rdir,
            completed,
            {
                "model": smodel.trainable_state_dict(model),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "rng": checkpoint.rng_states(),
            },
            keep_k=cfg.keep_k,
        )
    if completed % cfg.val_every == 0 or completed == cfg.total_steps:
        fig = render_val_grid(val_items, completed)
        fig.savefig(grid_dir / f"step_{completed:07d}.png", dpi=110, bbox_inches="tight")
        plt.show()
print("training loop finished")

In [ ]:
# The full training history: loss terms (log scale) and the LR schedule.
history = checkpoint.read_log(log_path)
fig = viz.plot_loss_curves(history, keys=("loss", "mse", "lpips", "mask"))
plt.show()

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.plot(history["step"], history["lr"], color=plt.get_cmap("tab10").colors[4])
ax.set_xlabel("step")
ax.set_ylabel("learning rate")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Post-run gallery on more designs than the loop's four.
gallery_items = [val_dataset[i] for i in range(min(6, len(val_dataset)))]
latest_step = checkpoint.find_latest(rdir)[1]
fig = render_val_grid(gallery_items, latest_step)
plt.show()

In [ ]:
# Before and after: the loop saved a validation grid every interval - the
# first and the last tell the story of the run.
from PIL import Image

grids = sorted(grid_dir.glob("step_*.png"))
if len(grids) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
    for ax, path in zip(axes, [grids[0], grids[-1]]):
        ax.imshow(Image.open(path))
        ax.set_title(path.stem)
        ax.axis("off")
    plt.show()
else:
    print(f"{len(grids)} grid(s) saved so far - train longer to compare")

## Reading the results

The validation grids are the honest signal: predictions should sharpen from gray blobs toward the ground-truth silhouettes as steps accumulate, and the loss terms should fall together (if LPIPS stalls while MSE falls, the model is matching pixels but not structure).

Notebook 06 measures this properly - 3D metrics against ground-truth meshes on the held-out test split, plus a comparison against the pretrained model given the dataset's own thumbnails.